<a href="https://colab.research.google.com/github/427paul/AI_Agent/blob/main/%5BBDA%5D_10%EC%A3%BC%EC%B0%A8_%EB%B3%B5%EC%8A%B5%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10주차 복습 과제: 하루스테이 예약 챗봇
## 🏨 시나리오: 숙박 예약 서비스 "하루스테이"

```
여러분은 숙박 예약 스타트업 '하루스테이'의 주니어 개발자입니다.
대표의 요청: "예약 상담 챗봇을 만들어주세요. 예약자 이름이랑 예약번호는
           한 번만 입력하면 대화 내내 기억해야 하고,
           문의가 길어져도 느려지면 안 됩니다."
```

실습에서 만든 위니브마켓 상담봇과 **구조가 동일**합니다.
빈칸(`# ??`)을 채우면서 각 코드가 왜 그렇게 생겼는지 다시 떠올려보세요.

| 실습 매핑 | 위니브마켓(실습) | 하루스테이(과제) |
| --- | --- | --- |
| 페르소나 | 위니봇 | 하루봇 |
| 식별 정보 | 고객 이름 + 주문번호 | 예약자 이름 + 예약번호 |
| 문의 예시 | 배송 지연 | 체크인 시간 변경 |

| Part | 내용 | 배점 |
| --- | --- | --- |
| A-1 | MessagesPlaceholder로 프롬프트 구성 | 15점 |
| A-2 | RunnableWithMessageHistory로 메모리 연결 | 15점 |
| A-3 | session_id로 다중 예약자 분리 | 20점 |
| A-4 | 요약 체인 직접 구성 | 20점 |
| B | 직접 구현 (옵션 선택) | 30점 |

> ⚠️ **시작 전 체크**: 아래 `0️⃣` 셀에서 API Key를 입력하세요.

---
## 0️⃣ 환경 설정

In [1]:
# 패키지 설치 (Colab 최초 1회)
!pip install -q langchain langchain-google-genai
print('✅ 설치 완료')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 1.7 MB/s eta 0:00:00
✅ 설치 완료


In [2]:
import os

# ──────────────────────────────────────────────
# ✏️  여기에 API Key 입력 (Google AI Studio 발급)
os.environ['GOOGLE_API_KEY'] = 'YOUR_GOOGLE_API'
# ──────────────────────────────────────────────

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.3,
)
print('✅ LLM 준비 완료 — 하루스테이 예약봇 개발 시작!')

✅ LLM 준비 완료 — 하루스테이 예약봇 개발 시작!


---
## 📋 Part A — 빈칸 채우기 (70점)

`# ??` 표시가 있는 부분을 채워주세요. 각 문제는 마지막에 `assert`로 자동 채점됩니다.

### A-1. MessagesPlaceholder로 프롬프트 구성 (15점)

**힌트**: 실습 셀 3️⃣를 참고하세요. `MessagesPlaceholder`는 대화 이력이 들어갈 자리를 예약합니다.
`variable_name`은 나중에 `RunnableWithMessageHistory`의 `history_messages_key`와 이름이 같아야 합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 하루스테이 예약 상담원입니다. 친절하게 응대하세요.'),
    # ?? : 대화 이력이 들어갈 자리를 예약하는 클래스를 사용하세요.
    #      variable_name은 'history'로 지정합니다.
    MessagesPlaceholder(variable_name="history"), #??
    ('human', '{input}'),
])

chain = prompt | llm

# ── 자동 채점 (수정하지 마세요) ──
placeholder_found = any(
    type(m).__name__ == 'MessagesPlaceholder' and getattr(m, 'variable_name', None) == 'history'
    for m in prompt.messages
)
assert placeholder_found, '❌ MessagesPlaceholder(variable_name="history")가 프롬프트에 없습니다.'
print('✅ A-1 통과!')

✅ A-1 통과!


### A-2. RunnableWithMessageHistory로 메모리 연결 (15점)

**힌트**: 실습 셀 4️⃣를 참고하세요. `input_messages_key`와 `history_messages_key`가
각각 어떤 역할을 하는지 떠올려보세요.

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",    # ?? : 현재 사용자 입력이 들어갈 키 이름 ('{input}'과 일치해야 함)
    history_messages_key="history",  # ?? : 대화 이력이 들어갈 키 이름 (A-1의 variable_name과 일치해야 함)
)

# ── 자동 채점 (수정하지 마세요) ──
config_test = {'configurable': {'session_id': 'test_a2'}}
r1 = chain_with_history.invoke({'input': '안녕하세요. 저는 박서연이고 예약번호는 HS-0001입니다.'}, config=config_test)
r2 = chain_with_history.invoke({'input': '제 이름이 뭐였죠?'}, config=config_test)

assert '서연' in r2.content, '❌ 메모리가 작동하지 않습니다. input_messages_key/history_messages_key를 다시 확인하세요.'
print('✅ A-2 통과!')
print('   [확인 응답]', r2.content[:60])

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


✅ A-2 통과!
   [확인 응답] 고객님의 성함은 **박서연** 고객님이십니다.

혹시 다른 문의사항이 있으실까요?


### A-3. session_id로 다중 예약자 분리 (20점)

**힌트**: 실습 셀 1️⃣1️⃣~1️⃣2️⃣를 참고하세요. `session_id`가 다르면 완전히 독립된 대화가 유지됩니다.
두 예약자(GUEST-A, GUEST-B)가 동시에 상담해도 정보가 섞이지 않아야 합니다.

In [ ]:
def consult(guest_id: str, message: str) -> str:
    """하루봇과 상담하는 함수. guest_id로 예약자를 구분합니다."""
    response = chain_with_history.invoke(
        {'input': message},
        # ?? : config 딕셔너리를 완성하세요.
        #      'configurable' 안에 'session_id' 키로 guest_id를 전달해야 합니다.
        config={"configurable" : {"session_id" : guest_id}},
    )
    return response.content

# 예약자 A
consult('GUEST-A', '안녕하세요. 박서연이고 예약번호는 HS-20241220-0007이에요.')
consult('GUEST-A', '체크인을 오후 3시로 변경하고 싶어요.')

# 예약자 B (A와 동시)
consult('GUEST-B', '안녕하세요. 최도윤입니다. 예약번호 HS-20241221-0015 취소하고 싶어요.')

# ── 자동 채점 (수정하지 마세요) ──
ra = consult('GUEST-A', '제 이름이랑 예약번호 확인해주세요.')
rb = consult('GUEST-B', '제 이름이랑 예약번호 확인해주세요.')

print('[GUEST-A 응답]', ra)
print('[GUEST-B 응답]', rb)
print()

assert '서연' in ra, '❌ GUEST-A 응답에 본인 이름이 없습니다.'
assert '도윤' in rb, '❌ GUEST-B 응답에 본인 이름이 없습니다.'

# 참고: 위 두 assert는 각자 '본인' 정보를 정확히 기억하는지 확인합니다.
# 두 응답을 직접 눈으로도 비교해보세요 — 서로의 예약번호가 섞여있지 않은지 확인하는 것이
# session_id 분리가 제대로 동작했는지 보는 가장 확실한 방법입니다.
print('✅ A-3 통과! 두 예약자의 대화가 정확히 분리되었습니다.')

[GUEST-A 응답] 네, 박서연 고객님! 고객님의 성함과 예약번호를 다시 한번 확인해 드리겠습니다.

고객님의 성함은 **박서연** 님이시며, 예약번호는 **HS-20241220-0007** 입니다.

정확하게 확인되셨을까요? 혹시 다른 문의사항이 있으시면 언제든지 말씀해주세요! 😊
[GUEST-B 응답] 네, 최도윤 고객님!

고객님의 성함과 예약번호는 제가 정확히 확인하고 있습니다.

*   **성함:** 최도윤 고객님
*   **예약번호:** HS-20241221-0015

다만, 고객님의 소중한 개인 정보 보호와 정확한 예약 취소 처리를 위해, 예약 시 등록하셨던 **휴대폰 번호 또는 이메일 주소** 중 하나를 알려주시면 감사하겠습니다. 이 정보가 확인되어야 본인 확인 후 안전하게 취소 절차를 진행할 수 있습니다.

번거로우시겠지만, 다시 한번 확인 부탁드립니다. 😊

✅ A-3 통과! 두 예약자의 대화가 정확히 분리되었습니다.


### A-4. 요약 체인 직접 구성 (20점)

**힌트**: 실습 셀 7️⃣를 참고하세요. `RunnableWithMessageHistory`는 이력을 저장만 할 뿐
요약 기능은 없습니다. '기존 요약 + 새 대화 → 새 요약'을 만드는 체인을 직접 구성해야 합니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

summary_prompt = ChatPromptTemplate.from_messages([
    ('system', '아래 예약 상담 요약과 새로운 대화를 한 문단으로 압축하세요. 예약자 이름, 예약번호, 문의 핵심은 반드시 남기세요.'),
    # ?? : human 메시지 템플릿을 작성하세요.
    #      {summary}, {user_input}, {ai_output} 세 변수를 모두 포함해야 합니다.
    #      (힌트: '[기존 요약]\n{summary}\n\n[새 대화]\n고객: {user_input}\n상담원: {ai_output}' 형태)
    ('human', '[기존 요약]\n{summary}\n\n[새 대화]\n고객: {user_input}\n상담원: {ai_output}'),
])

summary_chain = summary_prompt | llm | StrOutputParser()

def update_summary(summary: str, user_input: str, ai_output: str) -> str:
    return summary_chain.invoke({
        'summary': summary if summary else '(아직 대화 없음)',
        'user_input': user_input,
        'ai_output': ai_output,
    })

# ── 자동 채점 (수정하지 마세요) ──
test_summary = ''
test_summary = update_summary(test_summary, '저는 최도윤이고 예약번호 HS-0015입니다.', '안녕하세요, 최도윤 고객님!')
test_summary = update_summary(test_summary, '체크아웃을 1시간 늦추고 싶어요.', '네, 확인해 드리겠습니다.')

assert len(test_summary) > 0, '❌ 요약 결과가 비어 있습니다.'
assert '도윤' in test_summary or 'HS-0015' in test_summary, '❌ 핵심 정보(이름 또는 예약번호)가 요약에 남아있지 않습니다.'
print('✅ A-4 통과!')
print('   [생성된 요약]', test_summary)

✅ A-4 통과!
   [생성된 요약] 최도윤 고객이 예약번호 HS-0015로 상담을 시작했으며, 체크아웃을 1시간 늦추고 싶다고 문의했습니다. 상담원은 이를 확인해 주겠다고 답했습니다.


---
## 🚀 Part B — 직접 구현 (30점)

빈칸 채우기 없이, 아래 두 옵션 중 **하나를 선택**해 직접 구현하세요.
코드 스타일은 자유롭게 작성해도 됩니다.

### 옵션 선택

**옵션 1 — 예약 변경/취소 응대 보완**
하루봇의 시스템 프롬프트를 보완해서, 고객이 "변경"이나 "취소"를 언급하면
반드시 예약번호를 재확인하고 처리 절차를 안내하도록 만드세요.

**옵션 2 — 3명 이상 동시 예약자 처리**
`session_id`를 3개 이상 사용해 동시 상담 시나리오를 시뮬레이션하고,
전체 세션 현황(예약자별 대화 턴수, 최근 메시지)을 출력하는 코드를 작성하세요.

---

**선택한 옵션을 적어주세요:** `?? (옵션 1 또는 옵션 2)`

옵션 1

In [ ]:
# Part B 구현 공간
# 아래에 자유롭게 코드를 작성하세요.
# 시스템 프롬프트를 새로 정의하거나, 기존 chain_with_history를 재사용해도 됩니다.

# 예시 골격 (옵션 1을 고른 경우)
# HARUBOT_PROMPT_V2 = '''당신은 하루스테이의 예약 상담원 하루봇입니다.
#
# 핵심 규칙:
# 1. ...
# 2. 고객이 변경이나 취소를 언급하면 예약번호를 다시 확인하고 절차를 안내하세요.
# '''
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. LLM 및 세션 저장소 초기화
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)
option1_store = {}

def get_session_history_opt1(session_id: str):
    if session_id not in option1_store:
        option1_store[session_id] = InMemoryChatMessageHistory()
    return option1_store[session_id]


# 2. 변경/취소 응대 원칙을 보완한 시스템 프롬프트 설계
HARU_SYSTEM_PROMPT = """
당신은 예약 상담 전문 AI 어시스턴트 "하루봇"입니다.

[상담 및 예외 처리 원칙]
1. 고객이 예약 '변경' 또는 '취소'를 요청하거나 언급하는 경우, 반드시 다음 절차를 따르세요:
   - 기존 대화 맥락에서 고객이 '예약번호'를 이미 말했는지 먼저 확인합니다.
   - 만약 예약번호를 말하지 않았다면, 정중하게 "고객님, 처리를 위해 예약번호를 먼저 말씀해 주시겠어요?"라고 '예약번호 재확인'을 요청해야 합니다. 절대 예약번호 없이 절차를 가동하지 마세요.
   - 예약번호가 확인되면 변경/취소 처리 절차(확인, 소요 시간 등)를 안내하세요.
2. 답변은 친절하고 공손하게, 3문장 이내로 명확하게 작성하세요.
"""

# 3. LCEL 체인 조립 및 메시지 플레이스홀더 결합
prompt = ChatPromptTemplate.from_messages([
    ("system", HARU_SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="history"), # 이전 대화가 주입될 공간
    ("human", "{input}")
])

chain = prompt | llm | StrOutputParser()

harubot = RunnableWithMessageHistory(
    chain,
    get_session_history_opt1,
    input_messages_key="input",
    history_messages_key="history"
)




In [ ]:
# Part B 동작 확인
# 구현한 기능이 실제로 작동하는지 보여주는 테스트 코드를 작성하고 실행하세요.
# (출력 결과가 채점에 포함됩니다)

# ── 시뮬레이션 테스트 ──
def talk_to_haru(user_id: str, text: str):
    print(f"👤 고객: {text}")
    response = harubot.invoke(
        {"input": text},
        config={"configurable": {"session_id": user_id}}
    )
    print(f"🤖 하루봇: {response}\n")

print("--- [옵션 1] 예약 변경/취소 방어 로직 검증 ---")
# 시나리오: 예약번호를 주지 않고 예약을 취소해달라고 조르는 경우
talk_to_haru("user_01", "안녕하세요, 지난주에 한 예약을 취소하고 싶어서 연락드렸습니다.")
talk_to_haru("user_01", "아 참, 제 이름은 박선영입니다.") # 여전히 예약번호 누락 시 방어 작동 확인



--- [옵션 1] 예약 변경/취소 방어 로직 검증 ---
👤 고객: 안녕하세요, 지난주에 한 예약을 취소하고 싶어서 연락드렸습니다.
🤖 하루봇: 안녕하세요, 예약 취소 요청하셨군요. 처리를 위해 예약번호를 먼저 말씀해 주시겠어요?

👤 고객: 아 참, 제 이름은 박선영입니다.
🤖 하루봇: 고객님, 성함이 박선영이시군요. 예약 취소 처리를 위해 예약번호를 말씀해 주시면 감사하겠습니다.



옵션2

In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

# 1. 인프라 설정
prompt_opt2 = ChatPromptTemplate.from_messages([
    ("system", "당신은 호텔 예약 봇입니다. 예약자의 상황에 맞춰 친절하게 한 문장으로 답변하세요."),
    MessagesPlaceholder(variable_name="history"), # 대화 이력 삽입 위치 예약
    ("human", "{input}")
])

# ⚡ [수정] StrOutputParser를 추가하여 LLM의 출력을 깔끔한 문자열로 파싱합니다.
chain = prompt_opt2 | llm | StrOutputParser()

option2_store = {} # 멀티 세션 관리를 위한 인메모리 딕셔너리

def get_session_history_opt2(session_id: str):
    if session_id not in option2_store:
        option2_store[session_id] = InMemoryChatMessageHistory()
    return option2_store[session_id]

# 메시지 히스토리를 자동 관리하는 래퍼 체인 생성
multi_chatbot = RunnableWithMessageHistory(
    chain,
    get_session_history_opt2,
    input_messages_key="input",
    history_messages_key="history" # MessagesPlaceholder의 변수명과 일치[cite: 1]
)

def consult(guest_id: str, message: str) -> str:
    """하루봇과 상담하는 함수. guest_id로 예약자를 구분합니다."""
    response = multi_chatbot.invoke(
        {'input': message},
        config={"configurable": {"session_id": guest_id}} # 세션 분리를 위한 설정값 매핑[cite: 1]
    )
    # ⚡ [수정] chain의 끝에 StrOutputParser가 결합되었으므로 response 자체가 문자열(str)이 됩니다.[cite: 1]
    return response

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [4]:
# 2. 👥 동시 다발적 상담 시뮬레이션 실행 (3명 이상)
print("--- [옵션 2] 다중 세션 동시 상담 시뮬레이션 가동 ---")

# ⚡ [수정] 정의하신 consult 함수를 사용하여 시뮬레이션을 수행하도록 변경했습니다.
# 1턴 발생
consult("CUST_DOYUAN", "안녕하세요, 저는 최도윤이고 디럭스룸 예약 건 변경 원해요.")
consult("CUST_CHULSOO", "환불 규정이 어떻게 되나요? 예약자 김철수입니다.")
consult("CUST_MINSU", "박민수인데 내일 체크인 시간을 2시간 일찍 당길 수 있나요?")

# 특정 유저(최도윤, 김철수)만 추가 대화 진행 (턴 수 차별화)
consult("CUST_DOYUAN", "예약번호는 HS-7701 입니다.")
consult("CUST_CHULSOO", "결제는 카드로 처리했었습니다.")
consult("CUST_DOYUAN", "일정이 변경되어서 날짜를 하루 미루고 싶어요.")


# 3. 📊 전체 세션 현황 출력 및 모니터링 모듈
print("\n========================================================")
print("📊 [실시간 통합 대화 인프라 모니터링 시스템]")
print("========================================================")

for session_id, history_obj in option2_store.items():
    messages = history_obj.messages
    turn_count = len(messages) # HumanMessage + AIMessage의 총합 수[cite: 1]

    print(f"📍 세션 ID (고객 고유 키): {session_id}")
    print(f"   - 누적 대화 메시지 수: {turn_count}개 (약 {turn_count // 2}턴 진행)[cite: 1]")

    if turn_count > 0:
        last_msg = messages[-1] # 가장 마지막 AI 응답이나 Human 메시지 확보[cite: 1]
        print(f"   - 마지막 대화 유형: {type(last_msg).__name__}[cite: 1]")
        print(f"   - 최근 메시지 내용: \"{last_msg.content}\"[cite: 1]")
    else:
        print("   - 활성화된 대화 내역이 없습니다.[cite: 1]")
    print("-" * 56)

--- [옵션 2] 다중 세션 동시 상담 시뮬레이션 가동 ---

📊 [실시간 통합 대화 인프라 모니터링 시스템]
📍 세션 ID (고객 고유 키): CUST_DOYUAN
   - 누적 대화 메시지 수: 6개 (약 3턴 진행)[cite: 1]
   - 마지막 대화 유형: AIMessage[cite: 1]
   - 최근 메시지 내용: "알겠습니다. 기존 예약 날짜에서 하루 뒤로 변경해드리겠습니다. 혹시 기존 예약 날짜가 언제이신지 알려주시면 정확히 변경해드리겠습니다."[cite: 1]
--------------------------------------------------------
📍 세션 ID (고객 고유 키): CUST_CHULSOO
   - 누적 대화 메시지 수: 4개 (약 2턴 진행)[cite: 1]
   - 마지막 대화 유형: AIMessage[cite: 1]
   - 최근 메시지 내용: "네, 카드 결제 정보는 확인에 도움이 되지만, 예약 번호를 알려주시면 해당 예약의 정확한 환불 규정을 바로 찾아드릴 수 있습니다."[cite: 1]
--------------------------------------------------------
📍 세션 ID (고객 고유 키): CUST_MINSU
   - 누적 대화 메시지 수: 2개 (약 1턴 진행)[cite: 1]
   - 마지막 대화 유형: AIMessage[cite: 1]
   - 최근 메시지 내용: "박민수 고객님, 내일 체크인 시간을 2시간 앞당기는 것이 가능한지 확인해 드리겠습니다."[cite: 1]
--------------------------------------------------------


---
## 📝 회고 (필수 작성)

In [ ]:
review = """
[과제를 하면서 느낀 점]
- 실습과 다르게 헷갈렸던 부분: ??
- 가장 이해가 잘 된 개념: ??
- Part B에서 선택한 옵션과 이유: ??
"""
print(review)

---
## ✅ 제출 전 체크리스트

- [ ] API Key를 `YOUR_API_KEY`로 다시 바꿨나요?
- [ ] A-1 ~ A-4 모두 `✅ 통과!` 메시지를 확인했나요?
- [ ] Part B 옵션 중 하나를 선택해 구현하고 테스트 출력을 남겼나요?
- [ ] 회고를 작성했나요?
- [ ] 파일명을 `10주차_복습과제_이름.ipynb` 형식으로 바꿨나요?